# Publication figures — editable one graph at a time

Run the setup and data-preparation cells once, then edit and rerun any individual figure cell. Every metric uses completed, validation-selected checkpoints and the independent test split. The plotting palette is colour-blind safe; raw seed points use deterministic jitter.


In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import Image as DisplayImage, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

PAPER_ROOT = PROJECT_ROOT / 'publication_30seed_result'
TAXON_STAGE_ROOT = PAPER_ROOT
OUTPUT_DIR = PAPER_ROOT / 'publication_bundle' / 'figures'
SOURCE_ROOT = OUTPUT_DIR / 'figure_sources'
DATA_ROOT = PROJECT_ROOT.parent / 'petridish-worm-images'
VISUAL_MODEL = 'convnext_base'
SPECIES_ABLATION = 'Aporrectodea_longa'

from scripts import build_holdout_visual_notebook as figure_builder
importlib.reload(figure_builder)

def show(stem):
    path = OUTPUT_DIR / f'{stem}.png'
    if path.exists():
        display(DisplayImage(filename=str(path)))
    else:
        print(f'Not created; completed input rows are missing: {path}')


## Prepare the completed-run data

This cell reads results but does not train or submit anything.


In [ ]:
runs = figure_builder.collect_runs(PAPER_ROOT)
baseline_frame = figure_builder.prepare_baseline_frame(runs)
age_metrics, age_confusions = figure_builder.prepare_developmental_stage_diagnostics(baseline_frame)
visual_frames = figure_builder.prepare_convnext_visual_frames(runs, VISUAL_MODEL)
chance_reference = figure_builder.visual_uniform_chance_reference(runs, PROJECT_ROOT)
chance_row = chance_reference[chance_reference['task'].eq('mean')]
visual_chance = float(chance_row.iloc[0]['expected_uniform_macro_f1']) if not chance_row.empty else float('nan')
for frame in visual_frames.values():
    frame['chance'] = visual_chance

taxon_frame = figure_builder._paper_design_only(
    figure_builder._model_only(
        figure_builder.prepare_taxon_stage_holdout_frame(
            figure_builder.collect_adult_taxon_metrics(TAXON_STAGE_ROOT)
        ),
        figure_builder.TAXON_MODEL,
        context='data-ablation figures',
    ),
    require_loss_recipe=True,
)
paired = figure_builder.pair_taxon_metrics(taxon_frame)
paired, taxon_counts = figure_builder.attach_taxon_individual_counts(paired, PROJECT_ROOT)
species_paired = paired[paired.get('species', pd.Series(index=paired.index, dtype=str)).astype(str).eq(SPECIES_ABLATION)].copy() if not paired.empty else paired
biological_questions = figure_builder.prepare_biological_question_frame(taxon_frame, PROJECT_ROOT)


## Figure 1 — baseline scores and mean ConvNeXt confusion matrices


In [ ]:
figure_builder.save_baseline_overview(baseline_frame, OUTPUT_DIR, SOURCE_ROOT)
show('figure_01_all_models_all_tasks')


## Figure 1b — developmental-stage diagnostics


In [ ]:
figure_builder.save_developmental_stage_diagnostics(age_metrics, age_confusions, OUTPUT_DIR, SOURCE_ROOT)
show('figure_01b_developmental_stage_diagnostics')


## Figure 2 — visual ablations with a linear pixel axis


In [ ]:
figure_builder.save_convnext_visual_figure(visual_frames, OUTPUT_DIR, SOURCE_ROOT, model=VISUAL_MODEL, resolution_scale='linear', chance_reference=chance_reference)
show('figure_02_convnext_visual_ablation')


## Figure 2b — visual ablations with a log₂ pixel axis


In [ ]:
figure_builder.save_convnext_visual_figure(visual_frames, OUTPUT_DIR, SOURCE_ROOT, model=VISUAL_MODEL, resolution_scale='log2', chance_reference=chance_reference)
show('figure_02b_convnext_visual_ablation_resolution_log2')


## Figure 2c — mixed visual ablations as paired-seed comparisons


In [ ]:
figure_builder.save_mixed_visual_seed_figure(visual_frames.get('interaction', pd.DataFrame()), OUTPUT_DIR, SOURCE_ROOT, model=VISUAL_MODEL, chance_reference=chance_reference)
show('figure_02c_mixed_visual_seed_comparison')


## Figure 3 — configurable one-species ablation


In [ ]:
figure_builder.save_paired_estimation_figure(species_paired, OUTPUT_DIR, SOURCE_ROOT, name='figure_03_species_ablation', title=f"ConvNeXt-Base species ablation: {SPECIES_ABLATION.replace('_', ' ')} — independent test cohort", details={'species_ablation': SPECIES_ABLATION, 'model': figure_builder.TAXON_MODEL, 'split': 'test', 'cohort': figure_builder.TEST_COHORT})
show('figure_03_species_ablation')


## Figure 3b — target precision, recall and F1


In [ ]:
figure_builder.save_species_target_metric_figure(species_paired, OUTPUT_DIR, SOURCE_ROOT, species_ablation=SPECIES_ABLATION)
show('figure_03b_species_ablation_precision_recall_f1')


## Figure 4 — four biological transfer questions


In [ ]:
figure_builder.save_biological_question_figure(biological_questions, OUTPUT_DIR, SOURCE_ROOT)
show('figure_04_biological_transfer_questions')


## Figure 5 — raw recall margins for the selected species


In [ ]:
figure_builder.save_species_margin_figure(species_paired, OUTPUT_DIR, SOURCE_ROOT, species_ablation=SPECIES_ABLATION)
show('figure_05_species_ablation_raw_margins')


## Supplementary Figure 1 — every species-stage standardized effect


In [ ]:
figure_builder.save_all_species_effect_figure(paired, OUTPUT_DIR, SOURCE_ROOT)
show('supplementary_figure_01_all_species_effects')


## Supplementary Figure 2 — every species-stage raw margin


In [ ]:
figure_builder.save_all_species_margin_figure(paired, OUTPUT_DIR, SOURCE_ROOT)
show('supplementary_figure_02_all_species_raw_margins')


## Representative transformations


In [ ]:
figure_builder.save_representative_transformations(split_root=PROJECT_ROOT, data_root=DATA_ROOT, output_dir=OUTPUT_DIR, source_root=SOURCE_ROOT)
show('figure_07_representative_transformations')
